# Create wrf angle files 
Used for climatekitae programmatic rotation of wind vectors. The data used in this notebook are on Nicole's local drive. Hopefully this notebook doesn't need to be rerun again. 

Author: Nicole Keeney<br>
Creation Date: 01-31-2025 

In [1]:
import xarray as xr

In [2]:
# Coordinate files with the proper structure from climakitae
d01_coords = xr.open_dataset("data/d01_coords.nc")
d02_coords = xr.open_dataset("data/d02_coords.nc")
d03_coords = xr.open_dataset("data/d03_coords.nc")

In [4]:
# Get wrf angle files
d01_wrfinput = xr.open_dataset("data/wrfinput_d01")
d02_wrfinput = xr.open_dataset("data/wrfinput_d02")
d03_wrfinput = xr.open_dataset("data/wrfinput_d03")

In [5]:
def create_dataset(di_wrfinput, di_coords):
    """Create WRF wind dataset such that it can be used in climakitae easily
    Combines the WRF wind angles with the coordinates taken from WRF data
    The spatial resolution of di_wrfinput should match the spatial resolution of di_coords

    Parameters
    ----------
    di_wrfinput: xr.Dataset
        WRF angle dataset for a given resolution (d01, d02, or d03)
    di_coords: xr.Dataset
        WRF coordinate info for a given resolution (d01, d02, or d03)

    Returns
    -------
    di_wrfinput: xr.Dataset
    """

    di_wrfinput = di_wrfinput.isel(Time=0)
    di_wrfinput = di_wrfinput[["SINALPHA", "COSALPHA"]]
    di_wrfinput = di_wrfinput.rename(
        {"XLAT": "lat", "XLONG": "lon", "south_north": "y", "west_east": "x"}
    )
    # Add coordinates
    di_wrfinput = di_wrfinput.assign_coords({"x": di_coords.x, "y": di_coords.y})

    # Add grid mapping attribute so that rioxarray can find CRS 
    di_wrfinput.SINALPHA.attrs["grid_mapping"] = "Lambert_Conformal"
    di_wrfinput.COSALPHA.attrs["grid_mapping"] = "Lambert_Conformal"

    return di_wrfinput

In [6]:
# Generate datasets
d01_processed = create_dataset(d01_wrfinput, d01_coords)
d02_processed = create_dataset(d02_wrfinput, d02_coords)
d03_processed = create_dataset(d03_wrfinput, d03_coords)

In [ ]:
# Upload to s3
# mode = "w" will overwrite existing zarr with same name 
bucket_dir = "s3://cadcat/tmp/era/wrf" # Don't have trailing /
d01_processed.to_zarr(f"{bucket_dir}/wrf_angles_d01.zarr", mode="w")
d02_processed.to_zarr(f"{bucket_dir}/wrf_angles_d02.zarr", mode="w")
d03_processed.to_zarr(f"{bucket_dir}/wrf_angles_d03.zarr", mode="w")